In [ ]:
%matplotlib widget
# Boilerplate import code for all libraries
# Changes to the precision require re-loading the kernel and need to be done before any op uses them.
import warpSPHCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any) # precision: float16|half|float32|single|float64|double

import warpSPHCore as sph
from warpSPHCore.type_config import *
print(get_type_config()) # confirms active settings

# Initialize warp at this point
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available(): # set the TORCH_CUDA_ARCH_LIST environment variable to the compute capability of the GPU for faster compiles
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import warnings
from tqdm import TqdmExperimentalWarning
warnings.filterwarnings("ignore", category=TqdmExperimentalWarning)
from tqdm.autonotebook import tqdm

# final import blocks that are generic
import matplotlib.pyplot as plt
from torch.profiler import profile, record_function, ProfilerActivity
import numpy as np
import math
import shlex    
import subprocess
import shutil

# custom SPH libraries
from warpSPHIntegrators.integration import *
from warpSPHCore import *
from warpSPHPlotting import *

# This library
from warpSPH import *

# The case utilities that contain all the case setup functions for the various test cases
from warpSPH.caseUtils import *

In [ ]:
def buildSimulationConfig(args):
    nx = args.nx
    dim = 2
    L = args.L
    dx = L / nx
    band = args.band
    W = args.W
    n_h = args.n_h
    targetDt = args.targetDt


    gamma = 5/3
    rho0 = 1
    nu_visc = 0.0005
    freeSurface = True

    timestamp = getCurrentTimestamp()
    obstacleText = f'{args.obstacleType}_maxExtent{args.maxExtent}_aspectRatio{args.aspectRatio}_offsetX{args.offsetX}_offsetY{args.offsetY}'
    motionText = f'linearMotion{args.linearMotion}_angularMotion{args.angularMotion}_motionType{args.motionType}_motionFrequency{args.motionFrequency}_linearVelocityDirection{args.linearVelocityDirection[0]}_{args.linearVelocityDirection[1]}_linearVelocityMagnitude{args.linearVelocityMagnitude}_angularVelocityMagnitude{args.angularVelocityMagnitude}'
    caseName = f'{args.caseName}_{timestamp}_{nx}_{n_h}_{L}_{W}_{obstacleText}'

    extraData = {
        'nx': nx,
        'dim': dim,
        'L': L,
        'n_h': n_h,

        'gamma': gamma,
        'rho0': rho0,
        'nu_visc': nu_visc,
    }

    device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    dtype = get_torch_precision()


    domain = buildDomainDescription(L + dx * (band) * 2, dim, True, device, dtype)
    domain.min = torch.tensor([-W/2 - dx * (band), -L/2 - dx * (band)], device = device, dtype = dtype)
    domain.max = torch.tensor([W/2 + dx * (band), L/2 + dx * (band)], device = device, dtype = dtype)

    # Semi periodic
    if args.semiPeriodic:
        domain.min = torch.tensor([-W/2, -L/2 - dx * (band)], device = device, dtype = dtype)
        domain.max = torch.tensor([W/2, L/2 + dx * (band)], device = device, dtype = dtype)

    # Closed domain
    if args.fullyPeriodic:
        domain.min = torch.tensor([-W/2, -L/2], device = device, dtype = dtype)
        domain.max = torch.tensor([W/2, L/2], device = device, dtype = dtype)

    interiorDomain = buildDomainDescription(L, dim, False, device, dtype)
    interiorDomain.min = torch.tensor([-W/2, -L/2], device = device, dtype = dtype)
    interiorDomain.max = torch.tensor([W/2, L/2], device = device, dtype = dtype)


    config, integrator = buildConfig(
        domain = domain,
        dim = dim,
        kernel = KernelFunctions.Wendland4,
        targetNeighbors = n_h_to_nH(4, dim),
        supportMode = SupportScheme.KernelMeanSymmetric,
        gradientMode = GradientScheme.Difference,
        laplacianMode = LaplacianScheme.Brookshaw,
        integrationScheme = IntegrationSchemeType.rungeKutta2,
        samplingScheme = SamplingScheme.regular,
        device = device,
        dtype = dtype,
        dt = None,
        adaptiveDt = True,
        cflFactor=0.3,
    )
    config.nx = nx + band * 2
    config.dx = dx

    config.minDt = 1e-8
    # config.dx = L / (nx * 2)

    scheme = WeaklyCompressibleSPHScheme.deltaSPH
    bundle = buildScheme(scheme)
    SimulationSystem, SimulationState = bundle.SimulationSystem, bundle.SimulationState
    SimulationUpdate = bundle.SimulationUpdate
    SimulationConfig = bundle.SimulationConfig
    fn, export_fn, import_fn = bundle.stepFunction, bundle.exportFunction, bundle.importFunction


    schemeConfig = SimulationConfig()
    schemeConfig.surfaceDetectionConfig.active = freeSurface
    schemeConfig.bandwith = L / args.bandWidth / config.dx


    schemeConfig.gravityConfig.active = not args.disableGravity
    schemeConfig.gravityConfig.type = GravityType.Directional
    schemeConfig.gravityConfig.magnitude = args.gravityMagnitude
    schemeConfig.gravityConfig.origin = args.gravityDirection   

    return config, schemeConfig, integrator, SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn, domain, interiorDomain

In [ ]:
import argparse
parser = argparse.ArgumentParser(description='Run the dam break simulation with obstacle.')

parser.add_argument('--nx', type=int, default=128, help='Number of particles along the x-axis')
parser.add_argument('--markerSize', type=int, default=8, help='Size of the markers in the plot')
parser.add_argument('--plotWidth', type=int, default=28, help='Width of the plot in inches')
parser.add_argument('--n_h', type=int, default=4, help='Target number of neighbors')
parser.add_argument('--L', type=float, default=2.0, help='Length of the domain')
parser.add_argument('--W', type=float, default=4.0, help='Width of the domain')
parser.add_argument('--fillRatio', type=float, default=0.25, help='Fill ratio for the domain')

parser.add_argument('--timeLimit', type=float, default=4.0, help='Time limit for the simulation')
parser.add_argument('--enableFreestream', action='store_true', help='Enable freestream boundary conditions')
parser.add_argument('--forcingWidth', type=float, default=2.0/16.0, help='Width of the forcing region')
parser.add_argument('--freeStreamVelocity', type=float, default=1.0, help='Velocity of the free stream')
parser.add_argument('--band', type=int, default=5, help='Number of particle bands around the domain for boundary conditions')

parser.add_argument('--targetDt', type=float, default=0.0005, help='Target timestep for the simulation')

parser.add_argument('--obstacleActive', action='store_true', help='If set, an obstacle will be included in the simulation')
parser.add_argument('--obstacleType', type=str, default='circle', help='Type of obstacle to include (none, circle, ellipse, box, roundedBox, equilateralTriangle, hexagon, horseshoe, star, nacaXXXX)')
parser.add_argument('--maxExtent', type=float, default=0.25, help='Maximum extent of the obstacle')
parser.add_argument('--aoa', type=float, default=0.0, help='Angle of attack of the obstacle in degrees')
parser.add_argument('--aspectRatio', type=float, default=1.0, help='Aspect ratio of the obstacle (for ellipse)')
parser.add_argument('--offsetX', type=float, default=-0.0, help='X offset of the obstacle')
parser.add_argument('--offsetY', type=float, default=0.0, help='Y offset of the obstacle')
parser.add_argument('--mergeBoundaries', action='store_true', help='If set, the boundaries will be merged into a single boundary region')

parser.add_argument('--linearMotion', action='store_true', help='Enable linear motion of the obstacle')
parser.add_argument('--angularMotion', action='store_true', help='Enable angular motion of the obstacle')
# The motion can either be fixed, i.e., constantly spinning, or it can be a function of time, e.g., sinusoidal motion. The user can specify the type of motion using the --motionType argument.
parser.add_argument('--motionType', type=str, default='fixed', help='Type of motion for the obstacle (fixed, sinusoidal for now)')
parser.add_argument('--motionFrequency', type=float, default=1.0, help='Frequency of the motion for the obstacle')

parser.add_argument('--linearVelocityDirection', type=float, nargs=2, default=[0.0, 1.0], help='Direction of the linear motion (as a 2D vector)')
parser.add_argument('--linearVelocityMagnitude', type=float, default=0.5, help='Magnitude of the linear motion')
parser.add_argument('--angularVelocityMagnitude', type=float, default=1.0, help='Magnitude of the angular motion')

parser.add_argument('--disableGravity', action='store_true', help='Disable gravity in the simulation')
parser.add_argument('--gravityDirection', type=float, nargs=2, default=[0.0, -1.0], help='Direction of gravity (default: [0.0, -1.0])')
parser.add_argument('--gravityMagnitude', type=float, default=9.81, help='Magnitude of gravity (default: 9.81)')

parser.add_argument('--enableSloshing', action='store_true', help='Enable sloshing motion in the simulation')
parser.add_argument('--sloshingAmplitude', type=float, default=0.1, help='Amplitude of sloshing motion (default: 0.1)')
parser.add_argument('--sloshingFrequency', type=float, default=1.0, help='Frequency of sloshing motion (default: 1.0)')

parser.add_argument('--caseName', type=str, default='4-open-flow', help='Name of the case to run (default: 12-dambreak)')
parser.add_argument('--plot', action='store_true', help='Enable plotting of the simulation results')
parser.add_argument('--plotInterval', type=int, default=10, help='Interval for plotting (default: 10)')

parser.add_argument('--bandWidth', type=float, default=16.0, help='Width of the band for the noise function')

parser.add_argument('--semiPeriodic', action='store_true', help='Enable semi-periodic boundary conditions')
parser.add_argument('--fullyPeriodic', action='store_true', help='Enable fully periodic boundary conditions')
parser.add_argument('--fluidWidth', type=float, default=2.0, help='Width of the fluid region (default: 4.0)')


# args = parser.parse_args()

# cmd_args = '--plot --enableFreestream --timeLimit 5.0 --obstacleActive --obstacleType equilateralBottom --offsetX -1.5 --W 6.0 --fillRatio 1.0 -- semiPeriodic'
cmd_args = '--plot --enableFreestream --timeLimit 5.0 --obstacleActive --obstacleType equilateralBottom --offsetX -1.5 --W 6.0 --fillRatio 1.0 --fullyPeriodic'
# cmd_args = '--plot --enableFreestream --timeLimit 5.0 --obstacleActive --obstacleType equilateralBottom --offsetX -1.5 --W 4.0 --fillRatio 1.0'
import shlex
    

args = parser.parse_args(shlex.split(cmd_args))

In [ ]:
examples = []
timeLimit = 6.0
nx = 128
targetDt = 0.0005

examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleActive --obstacleType squareBottom --offsetX 1.0 --W 4.0 --fillRatio 0.3333333333333333 --fluidWidth 0.4166666666666667 --maxExtent 0.25 --aoa 0 --caseName dambreak --nx {nx} --targetDt {targetDt}')
examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleActive --obstacleType squareBottom --offsetX -1.5 --W 6.0 --fillRatio 0.25 --fluidWidth 1.0 --maxExtent 0.25 --aoa 0 --semiPeriodic --enableFreestream --freeStreamVelocity 1.0 --caseName openChannel --nx {nx} --targetDt {targetDt}')
examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleActive --obstacleType squareBottom --offsetX -1.5 --W 6.0 --fillRatio 1.0 --fluidWidth 1.0 --maxExtent 0.375 --aoa 0 --semiPeriodic --disableGravity --enableFreestream --freeStreamVelocity 1.0  --caseName semiPeriodic --nx {nx} --targetDt {targetDt}')
examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleActive --obstacleType squareMiddle --offsetX -1.5 --W 6.0 --fillRatio 1.0 --fluidWidth 1.0 --maxExtent 0.375 --aoa 0 --fullyPeriodic --disableGravity —enableFreestream —freeStreamVelocity 1.0 —caseName fullyPeriodic —nx {nx} —targetDt {targetDt}')
examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.5 --aoa 0 --caseName boundedRandom --nx {nx} --targetDt {targetDt} --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed 2187561599 --octaves 2 --baseFrequency 2  ')
examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.5 --aoa 0 --caseName boundedRandom_wObstacle  --obstacleActive --nx {nx} --targetDt {targetDt} --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed 2187561599 --octaves 2 --baseFrequency 2  ')
examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.5 --aoa 0 --caseName periodicRandom --nx {nx} --targetDt {targetDt} --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed 2187561599 --octaves 2 --baseFrequency 2  --fullyPeriodic')
examples.append(f'python generator.py --plot --timeLimit {timeLimit} --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.5 --aoa 0 --caseName periodicRandom_wObstacle  --obstacleActive --nx {nx} --targetDt {targetDt} --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed 2187561599 --octaves 2 --baseFrequency 2  --fullyPeriodic')
examples.append(f'python generator.py --plot --timeLimit {timeLimit*1.5} --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.5 --aoa 0 --caseName kolmogorov_wObstacle --nx {nx} --targetDt {targetDt} --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed 2187561599 --octaves 4 --baseFrequency 2 --obstacleActive --fullyPeriodic --enableKolmogorovForcing --kolmogorovForcingWavenumber 2 --noiseAmplitude 0.1')
examples.append(f'python generator.py --plot --timeLimit {timeLimit*1.5} --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.5 --aoa 0 --caseName kolmogorov --nx {nx} --targetDt {targetDt} --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed 2187561599 --octaves 4 --baseFrequency 2 --fullyPeriodic --enableKolmogorovForcing --kolmogorovForcingWavenumber 2 --noiseAmplitude 0.1')

os.makedirs('cases', exist_ok=True)
with open(f'./cases/examples.sh', 'w') as f:
    for example in examples:
        f.write(example + '\n')

In [ ]:
import numpy as np
random_seeds = np.random.randint(0, 2**32 - 1, size=4)
def build_periodic_command(seed, octaves, baseFrequency, obstacle, domainBoundary, caseName):
    return f'python generator.py --plot --timeLimit 6.0 --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.5 --aoa 0 --caseName {caseName} --nx 256 --targetDt 0.0002 --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed {seed} --octaves {octaves} --baseFrequency {baseFrequency} {"--obstacleActive" if obstacle else ""} {"--fullyPeriodic" if not domainBoundary else ""}'

random_flows = []

for domainBoundary in [True, False]:
    for seed in random_seeds:
        for octaves in [1, 2, 4]:
            for baseFrequency in [1, 2]:
                for obstacle in [False]:
                    random_flows.append(build_periodic_command(seed, octaves, baseFrequency, obstacle, domainBoundary, "boundedRandom" if domainBoundary else "periodicRandom"))
import os
os.makedirs('./cases', exist_ok=True)
with open('./cases/periodic.sh', 'w') as f:
    for flow in random_flows:
        f.write(flow + '\n')

for domainBoundary in [True, False]:
    for seed in random_seeds:
        for octaves in [1, 2, 4]:
            for baseFrequency in [1, 2]:
                for obstacle in [True]:
                    random_flows.append(build_periodic_command(seed, octaves, baseFrequency, obstacle, domainBoundary, "boundedRandom_wObstacle" if domainBoundary else "periodicRandom_wObstacle"))
import os
os.makedirs('./cases', exist_ok=True)
with open('./cases/periodic_wObstacle.sh', 'w') as f:
    for flow in random_flows:
        f.write(flow + '\n')

In [ ]:
import numpy as np
random_seeds = np.random.randint(0, 2**32 - 1, size=4)
def build_kolmogorov_command(seed, octaves, baseFrequency, obstacle, caseName, k):
    return f'python generator.py --plot --timeLimit 8.0 --obstacleType circleMiddle --offsetX 0.0 --W 2.0 --fillRatio 1.0 --fluidWidth 1.0  --maxExtent 0.4 --aoa 0 --caseName {caseName} --nx 256 --targetDt 0.0002 --enableNoise --disableGravity --markerSize 8 --plotWidth 20 --seed {seed} --octaves {octaves} --baseFrequency {baseFrequency} {"--obstacleActive" if obstacle else ""} --fullyPeriodic --enableKolmogorovForcing --kolmogorovForcingWavenumber {k} --noiseAmplitude 0.1'

kolmogorov_flows = []

for obstacle in [True, False]:
    for seed in random_seeds:
        for octaves in [1, 2, 4]:
            for baseFrequency in [1, 2]:
                for k in [1, 2, 4, 8]:
                    kolmogorov_flows.append(build_kolmogorov_command(seed, octaves, baseFrequency, obstacle, "kolmogorov_wObstacle" if obstacle else "kolmogorov_noObstacle", k))
import os
os.makedirs('./cases', exist_ok=True)
with open('./cases/kolmogorov.sh', 'w') as f:
    for flow in kolmogorov_flows:
        f.write(flow + '\n')

In [ ]:
nx = 256
targetDt = 0.0002

In [ ]:
# arguments for periodic flow

case = 'dambreak'
cmd_args = f'--plot --enableFreestream --timeLimit 6.0 --obstacleActive --obstacleType equilateralBottom --offsetX 0.5 --W 4.0 --fillRatio {1.0/3.0} --fluidWidth {5/4 * 1/3} --maxExtent 0.25'

args = parser.parse_args(shlex.split(cmd_args))

config, schemeConfig, integrator, SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn, domain, interiorDomain = buildSimulationConfig(args)
device = config.device
dtype = config.dtype

x = torch.linspace(domain.min[0], domain.max[0], config.nx, device = device, dtype = dtype)
y = torch.linspace(domain.min[1], domain.max[1], config.nx, device = device, dtype = dtype)
xx, yy = torch.meshgrid(x, y, indexing='ij')
points = torch.stack([xx.flatten(), yy.flatten()], dim=-1)

# args.fillRatio = 0.25
# args.maxExtent = 0.25
maxExtent = args.maxExtent

# angle = -30
# aspect = 3
# offset = maxExtent / aspect * np.sin(np.abs(angle) * np.pi / 180)
obstacle = {
        'maxExtent': args.maxExtent,
        'offsetX': args.offsetX,
        'offsetY': args.offsetY,
        'aspectRatio': args.aspectRatio,
        'obstacleType': args.obstacleType,
        'aoa': args.aoa
    }
# obstacle = obstacles['circleMiddle']
from utils import buildPresetObstacles, build_sdfs

# For the openflow case we want to run:
# maxExtent
testMatrixPeriodic = [
    ('equilateralBottom', 0, [maxExtent], [0.0, 0.5, 1.0]),
    ('triangleBottom', 0, [maxExtent], [0.0, 0.5, 1.0]),
    ('circleBottom', 0, [maxExtent], [0.0, 0.5, 1.0]),
    ('ellipsoidBottom', 0, [maxExtent], [0.0, 0.5, 1.0]),
    ('squareBottom', 0, [maxExtent], [0.0, 0.5, 1.0]),
    ('wallBottom', [-45, 0, 45], [maxExtent], [0.0, 0.5, 1.0]),
]

commands = []

# fillRatios = [1.0/3.0, 1/2, 2/3]
# fillWidths = [1/4 * 1/3, 3/4 * 1/3, 5/4 * 1/3]
fillRatios = [1.0/3.0, 1/2, 2/3]
fillWidths = [5/4 * 1/3, 3/4 * 1/3, 1/4 * 1/3]


for obstacleType, aoas, maxExtents, offsets in testMatrixPeriodic:
    for aoa in aoas if isinstance(aoas, list) else [aoas]:
        for offset in offsets if isinstance(offsets, list) else [offsets]:
            for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
                for fillRatio, fluidWidth in zip(fillRatios, fillWidths):
                    commands.append(f'python generator.py --plot --timeLimit 6.0 --obstacleActive --obstacleType {obstacleType} --offsetX {offset} --W {args.W} --fillRatio {fillRatio} --fluidWidth {fluidWidth} --maxExtent {maxExtent} --aoa {aoa} --caseName {case} --nx {nx} --targetDt {targetDt}')

for fillRatio in fillRatios:
    for fluidWidth in fillWidths:
        commands.append(f'python generator.py --plot --timeLimit 6.0 --W {args.W} --fillRatio {fillRatio} --fluidWidth {fluidWidth} --caseName {case} --nx {nx} --targetDt {targetDt}')

os.makedirs('cases', exist_ok=True)
with open(f'./cases/{case}.sh', 'w') as f:
    for command in commands:
        f.write(command + '\n')

os.makedirs(f'./cases/{case}', exist_ok=True)

# fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

# for obstacleType, aoas, maxExtents, offsets in testMatrixPeriodic:
#     for aoa in aoas if isinstance(aoas, list) else [aoas]:
#         for offset in offsets if isinstance(offsets, list) else [offsets]:
#             for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
#         # args.obstacleType = obstacleType
#         # args.aoa = aoa
#                 # obstacleType, aoas, maxExtents = testMatrixPeriodic[0]
#                 # aoa = aoas[0] if isinstance(aoas, list) else aoas

#                 # maxExtent = maxExtents[0]  # Use the first maxExtent value
#                 presets = buildPresetObstacles(maxExtent, offset, args.L, args.fillRatio, aoa)
#                 obstacle = presets.get(obstacleType)

#                 # presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
#                 # obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

#                 axis[0,0].cla()
#                 axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees, Max Extent: {maxExtent}, Offset: {offset}")

#                 regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(config, schemeConfig, args.band, args, domain, interiorDomain, obstacle)

#                 sdf, sdf_grad = domain_sdf(points)
#                 plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
#                 axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

#                 for ax in axis.flatten():
#                     ax.set_aspect('equal')
#                     ax.set_xlim(domain.min[0].item(), domain.max[0].item())
#                     ax.set_ylim(domain.min[1].item(), domain.max[1].item())

#                 # axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height', xmin=0.1, xmax=domain.max[0].item())
#                 fluidH = args.fillRatio * args.L
#                 fluidW = args.fluidWidth * args.W

#                 axis[0,0].add_artist(plt.Rectangle((interiorDomain.min[0].item(), interiorDomain.min[1].item()), fluidW, fluidH, linestyle='--', linewidth=1.5, label='Fluid Height', fill = False))
#                 # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

#                 fig.tight_layout()
#                 fig.savefig(f'./cases/{case}/{case}_{obstacleType}_aoa{aoa}_maxExtent{maxExtent}_offset{offset}.png', dpi=300)
#     #             break
#     #         break
#     #     break
#     # break

In [ ]:
# arguments for periodic flow

case = 'fullyPeriodic'
cmd_args = '--plot --enableFreestream --timeLimit 8.0 --plotWidth 32 --obstacleActive --obstacleType equilateralBottom --W 6.0 --fillRatio 1.0 --fullyPeriodic --fluidWidth 1.0'

args = parser.parse_args(shlex.split(cmd_args))

config, schemeConfig, integrator, SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn, domain, interiorDomain = buildSimulationConfig(args)
device = config.device
dtype = config.dtype

x = torch.linspace(domain.min[0], domain.max[0], config.nx, device = device, dtype = dtype)
y = torch.linspace(domain.min[1], domain.max[1], config.nx, device = device, dtype = dtype)
xx, yy = torch.meshgrid(x, y, indexing='ij')
points = torch.stack([xx.flatten(), yy.flatten()], dim=-1)

# args.fillRatio = 0.25
# args.maxExtent = 0.25
# maxExtent = args.maxExtent

# angle = -30
# aspect = 3
# offset = maxExtent / aspect * np.sin(np.abs(angle) * np.pi / 180)
# obstacle = {
#         'maxExtent': args.maxExtent,
#         'offsetX': args.offsetX,
#         'offsetY': args.offsetY,
#         'aspectRatio': args.aspectRatio,
#         'obstacleType': args.obstacleType,
#         'aoa': args.aoa
#     }
# obstacle = obstacles['circleMiddle']
from utils import buildPresetObstacles

# For the openflow case we want to run:
# maxExtent
testMatrix = [
    # ('equilateralBottom', 0, [maxExtent], [-1.5]),
    ('equilateralMiddle', [-90, 0, 90, 180], [maxExtent], [-1.5]),
    # ('equilateralTop', [-90, 0, 90, 180], [maxExtent], [-1.5]),
    # ('triangleBottom', 0, [maxExtent], [-1.5]),
    ('triangleMiddle', [-90, -45, 0], [maxExtent], [-1.5]),
    # ('triangleTop', [-90, -45, 0, 45, 90], [maxExtent], [-1.5]),
    # ('circleBottom', 0, [maxExtent], [-1.5]),
    ('circleMiddle', 0, [maxExtent], [-1.5]),
    # ('circleTop', 0, [maxExtent], [-1.5]),
    # ('ellipsoidBottom', 0, [maxExtent], [-1.5]),
    ('ellipsoidMiddle', [-90, -45, 0], [maxExtent], [-1.5]),
    # ('ellipsoidTop', [-90, -45, 0, 45], [maxExtent], [-1.5]),
    # ('squareBottom', 0, [maxExtent], [-1.5]),
    ('squareMiddle', [-45, -30, 0], [maxExtent*1.5], [-1.5]),
    # ('squareTop', [-45, -30, 0, 30], [maxExtent], [-1.5]),
    # ('wallBottom', [-45, -30, 0, 30, 45], [maxExtent], [-1.5]),
    ('wallMiddle', [-45, -30, 0], [maxExtent], [-1.5]),
    # ('wallTop', [-45, -30, 0, 30, 45], [maxExtent], [-1.5]),
]

# commands = []

# for obstacleType, aoas, maxExtents, offsets in testMatrix:
#     for aoa in aoas if isinstance(aoas, list) else [aoas]:
#         for offset in offsets if isinstance(offsets, list) else [offsets]:
#             for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
#                 commands.append(f'python generator.py --plot --timeLimit 8.0 --obstacleActive --obstacleType {obstacleType} --offsetX {offset} --W {args.W} --fillRatio {args.fillRatio} --fluidWidth 1.0 --maxExtent {maxExtent} --aoa {aoa} --fullyPeriodic --disableGravity --enableFreestream --freeStreamVelocity 1.0 --caseName {case} --nx {nx} --targetDt {targetDt}')

# os.makedirs('cases', exist_ok=True)
# with open(f'./cases/{case}.sh', 'w') as f:
#     for command in commands:
#         f.write(command + '\n')

# os.makedirs(f'./cases/{case}', exist_ok=True)


# fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

# for obstacleType, aoas, maxExtents, offsets in testMatrix:
#     for aoa in aoas if isinstance(aoas, list) else [aoas]:
#         for offset in offsets if isinstance(offsets, list) else [offsets]:
#             for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
# # for obstacleType, aoas, maxExtents in testMatrixPeriodic:
# #     for aoa in aoas if isinstance(aoas, list) else [aoas]:
# #         # args.obstacleType = obstacleType
# #         # args.aoa = aoa
# #         maxExtent = maxExtents[0]  # Use the first maxExtent value
#                 presets = buildPresetObstacles(maxExtent, offset, args.L, args.fillRatio, aoa)
#                 obstacle = presets.get(obstacleType)

#         # presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
#         # obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

#                 axis[0,0].cla()
#                 axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees")

#                 regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(config, schemeConfig, args.band, args, domain, interiorDomain, obstacle)

#                 sdf, sdf_grad = domain_sdf(points)
#                 plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
#                 axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

#                 for ax in axis.flatten():
#                     ax.set_aspect('equal')
#                     ax.set_xlim(domain.min[0].item(), domain.max[0].item())
#                     ax.set_ylim(domain.min[1].item(), domain.max[1].item())

#                 fluidH = args.fillRatio * args.L
#                 fluidW = args.fluidWidth * args.W

#                 axis[0,0].add_artist(plt.Rectangle((interiorDomain.min[0].item(), interiorDomain.min[1].item()), fluidW, fluidH, linestyle='--', linewidth=1.5, label='Fluid Height', fill = False))

#                 # axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height')
#                 # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

#                 fig.tight_layout()
#                 fig.savefig(f'./cases/{case}/{case}_{obstacleType}_aoa{aoa}.png', dpi=300)

In [ ]:
# arguments for periodic flow

case = 'semiPeriodic'
cmd_args = '--plot --enableFreestream --timeLimit 8.0 --plotWidth 32 --obstacleActive --obstacleType equilateralBottom --W 6.0 --fillRatio 1.0 --semiPeriodic'

args = parser.parse_args(shlex.split(cmd_args))

config, schemeConfig, integrator, SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn, domain, interiorDomain = buildSimulationConfig(args)
device = config.device
dtype = config.dtype

x = torch.linspace(domain.min[0], domain.max[0], config.nx, device = device, dtype = dtype)
y = torch.linspace(domain.min[1], domain.max[1], config.nx, device = device, dtype = dtype)
xx, yy = torch.meshgrid(x, y, indexing='ij')
points = torch.stack([xx.flatten(), yy.flatten()], dim=-1)

# args.fillRatio = 0.25
# args.maxExtent = 0.25
maxExtent = args.maxExtent

# angle = -30
# aspect = 3
# offset = maxExtent / aspect * np.sin(np.abs(angle) * np.pi / 180)
obstacle = {
        'maxExtent': args.maxExtent,
        'offsetX': args.offsetX,
        'offsetY': args.offsetY,
        'aspectRatio': args.aspectRatio,
        'obstacleType': args.obstacleType,
        'aoa': args.aoa
    }
# obstacle = obstacles['circleMiddle']
from utils import buildPresetObstacles

# For the openflow case we want to run:
# maxExtent
testMatrix = [
    ('equilateralBottom', 0, [maxExtent / 2], [-1.5]),
    ('equilateralMiddle', [-90, 0, 90], [maxExtent / 2], [-1.5]),
    # ('equilateralTop', [-90, 0, 90, 180], [maxExtent], [-1.5]),
    ('triangleBottom', 0, [maxExtent], [-1.5]),
    ('triangleMiddle', [-90, -45, 0], [maxExtent], [-1.5]),
    # ('triangleTop', [-90, -45, 0, 45, 90], [maxExtent], [-1.5]),
    ('circleBottom', 0, [maxExtent], [-1.5]),
    ('circleMiddle', 0, [maxExtent], [-1.5]),
    # ('circleTop', 0, [maxExtent], [-1.5]),
    ('ellipsoidBottom', 0, [maxExtent], [-1.5]),
    ('ellipsoidMiddle', [-90, -45, 0], [maxExtent], [-1.5]),
    # ('ellipsoidTop', [-90, -45, 0, 45], [maxExtent], [-1.5]),
    ('squareBottom', 0, [maxExtent/2], [-1.5]),
    ('squareMiddle', [-45, -30, 0], [maxExtent], [-1.5]),
    # ('squareTop', [-45, -30, 0, 30], [maxExtent], [-1.5]),
    ('wallBottom', [-45, -30, 0, 30, 45], [maxExtent], [-1.5]),
    ('wallMiddle', [-45, -30, 0], [maxExtent], [-1.5]),
    # ('wallTop', [-45, -30, 0, 30, 45], [maxExtent], [-1.5]),
]

commands = []
for obstacleType, aoas, maxExtents, offsets in testMatrix:
    for aoa in aoas if isinstance(aoas, list) else [aoas]:
        for offset in offsets if isinstance(offsets, list) else [offsets]:
            for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
                # for fillRatio, fluidWidth in zip(fillRatios, fillWidths):
                commands.append(f'python generator.py --plot --timeLimit 8.0 --obstacleActive --obstacleType {obstacleType} --offsetX {offset} --W {args.W} --fillRatio {args.fillRatio} --fluidWidth 1.0 --maxExtent {maxExtent} --aoa {aoa} --semiPeriodic --disableGravity --enableFreestream --freeStreamVelocity 1.0  --caseName {case} --nx {nx} --targetDt {targetDt}')

os.makedirs('cases', exist_ok=True)
with open(f'./cases/{case}.sh', 'w') as f:
    for command in commands:
        f.write(command + '\n')

os.makedirs(f'./cases/{case}', exist_ok=True)

# fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

# for obstacleType, aoas, maxExtents, offsets in testMatrix:
#     for aoa in aoas if isinstance(aoas, list) else [aoas]:
#         for offset in offsets if isinstance(offsets, list) else [offsets]:
#             for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
# # for obstacleType, aoas, maxExtents in testMatrixPeriodic:
# #     for aoa in aoas if isinstance(aoas, list) else [aoas]:
# #         # args.obstacleType = obstacleType
# #         # args.aoa = aoa
# #         maxExtent = maxExtents[0]  # Use the first maxExtent value
#                 presets = buildPresetObstacles(maxExtent, offset, args.L, args.fillRatio, aoa)
#                 obstacle = presets.get(obstacleType)

#         # presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
#         # obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

#                 axis[0,0].cla()
#                 axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees")

#                 regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(config, schemeConfig, args.band, args, domain, interiorDomain, obstacle)

#                 sdf, sdf_grad = domain_sdf(points)
#                 plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
#                 axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

#                 for ax in axis.flatten():
#                     ax.set_aspect('equal')
#                     ax.set_xlim(domain.min[0].item(), domain.max[0].item())
#                     ax.set_ylim(domain.min[1].item(), domain.max[1].item())

#                 fluidH = args.fillRatio * args.L
#                 fluidW = args.fluidWidth * args.W

#                 axis[0,0].add_artist(plt.Rectangle((interiorDomain.min[0].item(), interiorDomain.min[1].item()), fluidW, fluidH, linestyle='--', linewidth=1.5, label='Fluid Height', fill = False))

#                 # axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height')
#                 # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

#                 fig.tight_layout()
#                 fig.savefig(f'./cases/{case}/{case}_{obstacleType}_aoa{aoa}.png', dpi=300)

In [ ]:
# arguments for periodic flow

case = 'openChannel'
cmd_args = '--plot --enableFreestream --timeLimit 8.0 --plotWidth 32 --obstacleActive --obstacleType equilateralBottom --W 6.0 --fillRatio 0.25 --semiPeriodic'

args = parser.parse_args(shlex.split(cmd_args))

config, schemeConfig, integrator, SimulationSystem, SimulationState, SimulationConfig, SimulationUpdate, fn, export_fn, import_fn, domain, interiorDomain = buildSimulationConfig(args)
device = config.device
dtype = config.dtype

x = torch.linspace(domain.min[0], domain.max[0], config.nx, device = device, dtype = dtype)
y = torch.linspace(domain.min[1], domain.max[1], config.nx, device = device, dtype = dtype)
xx, yy = torch.meshgrid(x, y, indexing='ij')
points = torch.stack([xx.flatten(), yy.flatten()], dim=-1)

# args.fillRatio = 0.25
# args.maxExtent = 0.25
maxExtent = args.maxExtent

# angle = -30
# aspect = 3
# offset = maxExtent / aspect * np.sin(np.abs(angle) * np.pi / 180)
obstacle = {
        'maxExtent': args.maxExtent,
        'offsetX': args.offsetX,
        'offsetY': args.offsetY,
        'aspectRatio': args.aspectRatio,
        'obstacleType': args.obstacleType,
        'aoa': args.aoa
    }
# obstacle = obstacles['circleMiddle']
from utils import buildPresetObstacles

# For the openflow case we want to run:
# maxExtent
testMatrix = [
    ('equilateralBottom', 0, [maxExtent], [-1.5]),
    ('equilateralMiddle', [0, 180], [maxExtent], [-1.5]),
    ('equilateralTop', [-90, 0, 90, 180], [maxExtent], [-1.5]),
    ('triangleBottom', 0, [maxExtent], [-1.5]),
    ('triangleMiddle', [-90, -45, 0, 45, 90], [maxExtent], [-1.5]),
    ('triangleTop', [-90, -45, 0, 45, 90], [maxExtent], [-1.5]),
    ('circleBottom', 0, [maxExtent], [-1.5]),
    ('circleMiddle', 0, [maxExtent], [-1.5]),
    ('circleTop', 0, [maxExtent], [-1.5]),
    ('ellipsoidBottom', 0, [maxExtent], [-1.5]),
    ('ellipsoidMiddle', [-90, -45, 0, 45], [maxExtent], [-1.5]),
    ('ellipsoidTop', [-90, -45, 0, 45], [maxExtent], [-1.5]),
    ('squareBottom', 0, [maxExtent], [-1.5]),
    ('squareMiddle', [-45, 0], [maxExtent*1.5], [-1.5]),
    ('squareTop', [-45, 0], [maxExtent], [-1.5]),
    ('wallBottom', [-45, 0, 45], [maxExtent], [-1.5]),
    ('wallMiddle', [-45, 0, 45], [maxExtent], [-1.5]),
    ('wallTop', [-45, 0, 45], [maxExtent], [-1.5]),
]


commands = []
for obstacleType, aoas, maxExtents, offsets in testMatrix:
    for aoa in aoas if isinstance(aoas, list) else [aoas]:
        for offset in offsets if isinstance(offsets, list) else [offsets]:
            for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
                # for fillRatio, fluidWidth in zip(fillRatios, fillWidths):
                commands.append(f'python generator.py --plot --timeLimit 8.0 --obstacleActive --obstacleType {obstacleType} --offsetX {offset} --W {args.W} --fillRatio {args.fillRatio} --fluidWidth 1.0 --maxExtent {maxExtent} --aoa {aoa} --semiPeriodic --enableFreestream --freeStreamVelocity 1.0 --caseName {case} --nx {nx} --targetDt {targetDt}')

os.makedirs('cases', exist_ok=True)
with open(f'./cases/{case}.sh', 'w') as f:
    for command in commands:
        f.write(command + '\n')

os.makedirs(f'./cases/{case}', exist_ok=True)

# fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

# for obstacleType, aoas, maxExtents, offsets in testMatrix:
#     for aoa in aoas if isinstance(aoas, list) else [aoas]:
#         for offset in offsets if isinstance(offsets, list) else [offsets]:
#             for maxExtent in maxExtents if isinstance(maxExtents, list) else [maxExtents]:
# # for obstacleType, aoas, maxExtents in testMatrixPeriodic:
# #     for aoa in aoas if isinstance(aoas, list) else [aoas]:
# #         # args.obstacleType = obstacleType
# #         # args.aoa = aoa
# #         maxExtent = maxExtents[0]  # Use the first maxExtent value
#                 presets = buildPresetObstacles(maxExtent, offset, args.L, args.fillRatio, aoa)
#                 obstacle = presets.get(obstacleType)

#         # presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
#         # obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

#                 axis[0,0].cla()
#                 axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees")

#                 regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(config, schemeConfig, args.band, args, domain, interiorDomain, obstacle)

#                 sdf, sdf_grad = domain_sdf(points)
#                 plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
#                 axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

#                 for ax in axis.flatten():
#                     ax.set_aspect('equal')
#                     ax.set_xlim(domain.min[0].item(), domain.max[0].item())
#                     ax.set_ylim(domain.min[1].item(), domain.max[1].item())

#                 fluidH = args.fillRatio * args.L
#                 fluidW = args.fluidWidth * args.W

#                 axis[0,0].add_artist(plt.Rectangle((interiorDomain.min[0].item(), interiorDomain.min[1].item()), fluidW, fluidH, linestyle='--', linewidth=1.5, label='Fluid Height', fill = False))

#                 # axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height')
#                 # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

#                 fig.tight_layout()
#                 fig.savefig(f'./cases/{case}/{case}_{obstacleType}_aoa{aoa}.png', dpi=300)

In [ ]:
# fillRatio = args.fillRatio

# fillRatio = 1

# fillHeight = fillRatio * args.L
# offsetX = args.offsetX
# maxExtent = 0.5
# domainL = args.L / 2
# print(f"maxExtent: {maxExtent}, fillHeight: {fillHeight}, domainL: {domainL}")

# angle = args.aoa
# angle = 30

In [ ]:
args.fillRatio = 0.25
args.maxExtent = 0.25
maxExtent = args.maxExtent

# angle = -30
# aspect = 3
# offset = maxExtent / aspect * np.sin(np.abs(angle) * np.pi / 180)
obstacle = {
        'maxExtent': args.maxExtent,
        'offsetX': args.offsetX,
        'offsetY': args.offsetY,
        'aspectRatio': args.aspectRatio,
        'obstacleType': args.obstacleType,
        'aoa': args.aoa
    }
# obstacle = obstacles['circleMiddle']
from utils import buildPresetObstacles

# For the openflow case we want to run:
# maxExtent
testMatrixOpenChannel = [
    ('equilateralBottom', 0, [maxExtent]),
    ('equilateralMiddle', [0, 180], [maxExtent]),
    ('equilateralTop', [-90, 0, 90, 180], [maxExtent]),
    ('triangleBottom', 0, [maxExtent]),
    ('triangleMiddle', [-90, -45, 0, 45, 90], [maxExtent]),
    ('triangleTop', [-90, -45, 0, 45, 90], [maxExtent]),
    ('circleBottom', 0, [maxExtent]),
    ('circleMiddle', 0, [maxExtent]),
    ('circleTop', 0, [maxExtent]),
    ('ellipsoidBottom', 0, [maxExtent]),
    ('ellipsoidMiddle', [-90, -45, 0, 45], [maxExtent]),
    ('ellipsoidTop', [-90, -45, 0, 45], [maxExtent]),
    ('squareBottom', 0, [maxExtent]),
    ('squareMiddle', [-45, -30, 0, 30], [maxExtent*1.5]),
    ('squareTop', [-45, -30, 0, 30], [maxExtent]),
    ('wallBottom', [-45, -30, 0, 30, 45], [maxExtent]),
    ('wallMiddle', [-45, -30, 0, 30, 45], [maxExtent]),
    ('wallTop', [-45, -30, 0, 30, 45], [maxExtent]),
]


os.makedirs(f'./cases/openChannel', exist_ok=True)

fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

for obstacleType, aoas, maxExtents in testMatrixOpenChannel:
    for aoa in aoas if isinstance(aoas, list) else [aoas]:
        # args.obstacleType = obstacleType
        # args.aoa = aoa
        maxExtent = maxExtents[0]  # Use the first maxExtent value
        presets = buildPresetObstacles(maxExtent, -args.W / 4, args.L, args.fillRatio, aoa)
        obstacle = presets.get(obstacleType)

# presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
# obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

        axis[0,0].cla()
        axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees")

        regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(args, domain, interiorDomain, obstacle)

        sdf, sdf_grad = domain_sdf(points)
        plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
        axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

        for ax in axis.flatten():
            ax.set_aspect('equal')
            ax.set_xlim(domain.min[0].item(), domain.max[0].item())
            ax.set_ylim(domain.min[1].item(), domain.max[1].item())

        axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height')
        # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

        fig.tight_layout()
        fig.savefig(f'./cases/openChannel/openChannel_{obstacleType}_aoa{aoa}.png', dpi=300)

In [ ]:
args.fillRatio = 0.25
args.maxExtent = 0.25
maxExtent = args.maxExtent

# angle = -30
# aspect = 3
# offset = maxExtent / aspect * np.sin(np.abs(angle) * np.pi / 180)
obstacle = {
        'maxExtent': args.maxExtent,
        'offsetX': args.offsetX,
        'offsetY': args.offsetY,
        'aspectRatio': args.aspectRatio,
        'obstacleType': args.obstacleType,
        'aoa': args.aoa
    }
# obstacle = obstacles['circleMiddle']
from utils import buildPresetObstacles

# For the openflow case we want to run:
# maxExtent
testMatrixOpenChannel = [
    # ('equilateralBottom', 0, [maxExtent]),
    # ('equilateralMiddle', [0, 180], [maxExtent]),
    # ('equilateralTop', [-90, 0, 90, 180], [maxExtent]),
    # ('triangleBottom', 0, [maxExtent]),
    # ('triangleMiddle', [-90, -45, 0, 45, 90], [maxExtent]),
    # ('triangleTop', [-90, -45, 0, 45, 90], [maxExtent]),
    # ('circleBottom', 0, [maxExtent]),
    # ('circleMiddle', 0, [maxExtent]),
    # ('circleTop', 0, [maxExtent]),
    # ('ellipsoidBottom', 0, [maxExtent]),
    # ('ellipsoidMiddle', [-90, -45, 0, 45], [maxExtent]),
    # ('ellipsoidTop', [-90, -45, 0, 45], [maxExtent]),
    ('squareBottom', 0, [maxExtent]),
    ('squareMiddle', [-45, -30, 0, 30], [maxExtent*1.5]),
    # ('squareTop', [-45, -30, 0, 30], [maxExtent]),
    # ('wallBottom', [-45, -30, 0, 30, 45], [maxExtent]),
    # ('wallMiddle', [-45, -30, 0, 30, 45], [maxExtent]),
    # ('wallTop', [-45, -30, 0, 30, 45], [maxExtent]),
]


# os.makedirs(f'./cases/openChannel', exist_ok=True)

# fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

# for obstacleType, aoas, maxExtents in testMatrixOpenChannel:
#     for aoa in aoas if isinstance(aoas, list) else [aoas]:
#         # args.obstacleType = obstacleType
#         # args.aoa = aoa
#         maxExtent = maxExtents[0]  # Use the first maxExtent value
#         presets = buildPresetObstacles(maxExtent, -args.W / 4, args.L, args.fillRatio, aoa)
#         obstacle = presets.get(obstacleType)

# # presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
# # obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

#         axis[0,0].cla()
#         axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees")

#         regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(args, domain, interiorDomain, obstacle)

#         sdf, sdf_grad = domain_sdf(points)
#         plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
#         axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

#         for ax in axis.flatten():
#             ax.set_aspect('equal')
#             ax.set_xlim(domain.min[0].item(), domain.max[0].item())
#             ax.set_ylim(domain.min[1].item(), domain.max[1].item())

#         axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height')
#         # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

#         fig.tight_layout()
#         fig.savefig(f'./cases/openChannel/openChannel_{obstacleType}_aoa{aoa}.png', dpi=300)

In [ ]:
args.fillRatio = 0.25
args.maxExtent = 0.25
maxExtent = args.maxExtent

# angle = -30
# aspect = 3
# offset = maxExtent / aspect * np.sin(np.abs(angle) * np.pi / 180)
obstacle = {
        'maxExtent': args.maxExtent,
        'offsetX': args.offsetX,
        'offsetY': args.offsetY,
        'aspectRatio': args.aspectRatio,
        'obstacleType': args.obstacleType,
        'aoa': args.aoa
    }
# obstacle = obstacles['circleMiddle']
from utils import buildPresetObstacles

# For the openflow case we want to run:
# maxExtent
testMatrixOpenChannel = [
    # ('equilateralBottom', 0, [maxExtent]),
    # ('equilateralMiddle', [0, 180], [maxExtent]),
    # ('equilateralTop', [-90, 0, 90, 180], [maxExtent]),
    # ('triangleBottom', 0, [maxExtent]),
    # ('triangleMiddle', [-90, -45, 0, 45, 90], [maxExtent]),
    # ('triangleTop', [-90, -45, 0, 45, 90], [maxExtent]),
    # ('circleBottom', 0, [maxExtent]),
    # ('circleMiddle', 0, [maxExtent]),
    # ('circleTop', 0, [maxExtent]),
    # ('ellipsoidBottom', 0, [maxExtent]),
    # ('ellipsoidMiddle', [-90, -45, 0, 45], [maxExtent]),
    # ('ellipsoidTop', [-90, -45, 0, 45], [maxExtent]),
    ('squareBottom', 0, [maxExtent]),
    ('squareMiddle', [-45, -30, 0, 30], [maxExtent*1.5]),
    # ('squareTop', [-45, -30, 0, 30], [maxExtent]),
    # ('wallBottom', [-45, -30, 0, 30, 45], [maxExtent]),
    # ('wallMiddle', [-45, -30, 0, 30, 45], [maxExtent]),
    # ('wallTop', [-45, -30, 0, 30, 45], [maxExtent]),
]


os.makedirs(f'./cases/openChannel', exist_ok=True)

fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

for obstacleType, aoas, maxExtents in testMatrixOpenChannel:
    for aoa in aoas if isinstance(aoas, list) else [aoas]:
        # args.obstacleType = obstacleType
        # args.aoa = aoa
        maxExtent = maxExtents[0]  # Use the first maxExtent value
        presets = buildPresetObstacles(maxExtent, -args.W / 4, args.L, args.fillRatio, aoa)
        obstacle = presets.get(obstacleType)

# presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
# obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

        axis[0,0].cla()
        axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees")

        regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(config, schemeConfig, args.band, args, domain, interiorDomain, obstacle)

        sdf, sdf_grad = domain_sdf(points)
        plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
        axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

        for ax in axis.flatten():
            ax.set_aspect('equal')
            ax.set_xlim(domain.min[0].item(), domain.max[0].item())
            ax.set_ylim(domain.min[1].item(), domain.max[1].item())

        axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height')
        # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

        fig.tight_layout()
        # fig.savefig(f'./cases/openChannel/openChannel_{obstacleType}_aoa{aoa}.png', dpi=300)
        break
    break

In [ ]:
args.fillRatio = 1.0
args.maxExtent = 0.5
maxExtent = args.maxExtent

testMatrixFullChannel = [
    ('equilateralBottom', 0, [maxExtent / 2]),
    ('equilateralMiddle', [-90, 0, 90, 180], [maxExtent / 2]),
    # ('equilateralTop', [-90, 0, 90, 180], [maxExtent]),
    ('triangleBottom', 0, [maxExtent]),
    ('triangleMiddle', [-90, -45, 0, 45, 90], [maxExtent]),
    # ('triangleTop', [-90, -45, 0, 45, 90], [maxExtent]),
    ('circleBottom', 0, [maxExtent]),
    ('circleMiddle', 0, [maxExtent]),
    # ('circleTop', 0, [maxExtent]),
    ('ellipsoidBottom', 0, [maxExtent]),
    ('ellipsoidMiddle', [-90, -45, 0, 45], [maxExtent]),
    # ('ellipsoidTop', [-90, -45, 0, 45], [maxExtent]),
    ('squareBottom', 0, [maxExtent/2]),
    ('squareMiddle', [-45, -30, 0, 30], [maxExtent]),
    # ('squareTop', [-45, -30, 0, 30], [maxExtent]),
    ('wallBottom', [-45, -30, 0, 30, 45], [maxExtent]),
    ('wallMiddle', [-45, -30, 0, 30, 45], [maxExtent]),
    # ('wallTop', [-45, -30, 0, 30, 45], [maxExtent]),
]

os.makedirs(f'./cases/fullChannel', exist_ok=True)

fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

for obstacleType, aoas, maxExtents in testMatrixFullChannel:
    for aoa in aoas if isinstance(aoas, list) else [aoas]:
        # args.obstacleType = obstacleType
        # args.aoa = aoa
        maxExtent = maxExtents[0]  # Use the first maxExtent value
        presets = buildPresetObstacles(maxExtent, -args.W / 4, args.L, args.fillRatio, aoa)
        obstacle = presets.get(obstacleType)

# presets = buildPresetObstacles(args.maxExtent, -args.W / 4, args.L, args.fillRatio, args.aoa)
# obstacle = presets.get(args.obstacleType) if args.obstacleType in presets else obstacle

        axis[0,0].cla()
        axis[0,0].set_title(f"Obstacle: {obstacleType}, AOA: {aoa} degrees")

        regions, fluid_sdf, domain_sdf, obstacle_sdf = build_sdfs(args, domain, interiorDomain, obstacle)

        sdf, sdf_grad = domain_sdf(points)
        plotRegions(regions, axis[0,0], plotFluid = False, plotParticles = False)
        axis[0,0].contourf(xx.cpu().numpy(), yy.cpu().numpy(), sdf.cpu().numpy().reshape(config.nx, config.nx), levels=100, cmap='RdBu_r', alpha=0.5,vmin=-0.125, vmax=0.125)

        for ax in axis.flatten():
            ax.set_aspect('equal')
            ax.set_xlim(domain.min[0].item(), domain.max[0].item())
            ax.set_ylim(domain.min[1].item(), domain.max[1].item())

        axis[0,0].axhline(y = args.fillRatio * args.L - args.L/2, color = 'black', linestyle = '--', linewidth = 1.5, label = 'Fluid Height')
        # print(f"Fluid Height: {args.fillRatio * args.L - args.L/2}, Fill Ratio: {args.fillRatio}, L: {args.L}")

        fig.tight_layout()
        fig.savefig(f'./cases/fullChannel/fullChannel_{obstacleType}_aoa{aoa}.png', dpi=300)

In [ ]:
fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)
plotRegions(regions, axis[0,0], plotFluid = True, plotParticles = True)

rectangle = plt.Rectangle((domain.min[0].cpu().numpy(), domain.min[1].cpu().numpy()), domain.max[0].cpu().numpy() - domain.min[0].cpu().numpy(), domain.max[1].cpu().numpy() - domain.min[1].cpu().numpy(), fill=False, color='black', lw=2)
axis[0,0].add_patch(rectangle)

rectangleInterior = plt.Rectangle((interiorDomain.min[0].cpu().numpy(), interiorDomain.min[1].cpu().numpy()), interiorDomain.max[0].cpu().numpy() - interiorDomain.min[0].cpu().numpy(), interiorDomain.max[1].cpu().numpy() - interiorDomain.min[1].cpu().numpy(), fill=False, color='red', lw=2)
axis[0,0].add_patch(rectangleInterior)

import matplotlib.patches as patches

for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(domain.min[0].item(), domain.max[0].item())
    ax.set_ylim(domain.min[1].item(), domain.max[1].item())

fig.tight_layout()

In [ ]:

compressibleSystem = initializeWeaklyCompressibleSimulation(regions, config, schemeConfig, SimulationSystem, SimulationState, verbose = True)

In [ ]:
fig, axis = plt.subplots(1, 1, figsize=(15, 6), squeeze=False)

kinds = compressibleSystem.state.kinds
axis[0,0].scatter(compressibleSystem.state.positions[kinds == 0,0].cpu().numpy(), compressibleSystem.state.positions[kinds == 0,1].cpu().numpy(), s=1, color='blue', alpha=0.5)

axis[0,0].scatter(compressibleSystem.state.positions[kinds == 1,0].cpu().numpy(), compressibleSystem.state.positions[kinds == 1,1].cpu().numpy(), s=1, color='red', alpha=0.5)

axis[0,0].scatter(compressibleSystem.state.positions[kinds == 2,0].cpu().numpy(), compressibleSystem.state.positions[kinds == 2,1].cpu().numpy(), s=1, color='green', alpha=0.5)

for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(domain.min[0].item(), domain.max[0].item())
    ax.set_ylim(domain.min[1].item(), domain.max[1].item())

In [ ]:

t = torch.tensor(0, device = device, dtype = dtype)

obstacleLinearVelocity = torch.tensor(args.linearVelocityDirection, device = device, dtype = dtype) * args.linearVelocityMagnitude
obstacleAngularVelocity = args.angularVelocityMagnitude
obstacleMotionType = args.motionType
obstacleMotionFrequency = args.motionFrequency
print(f"Obstacle motion type: {obstacleMotionType}, linear velocity: {obstacleLinearVelocity}, angular velocity: {obstacleAngularVelocity}, motion frequency: {obstacleMotionFrequency}")
if obstacleMotionType == 'fixed':
    linearVelocity = obstacleLinearVelocity#[None,:]
    angularVelocity = torch.tensor(obstacleAngularVelocity, device = device, dtype = dtype)
elif obstacleMotionType == 'sinusoidal':
    linearVelocity =  obstacleLinearVelocity * torch.cos(t * np.pi * obstacleMotionFrequency)#[:,None]
    angularVelocity = obstacleAngularVelocity * torch.cos(t * np.pi * obstacleMotionFrequency) 
else:
    raise ValueError(f"Unknown motion type: {obstacleMotionType}")

if args.linearMotion == True:
    config.rigidBodies[0].linearVelocity = linearVelocity
if args.angularMotion == True:
    config.rigidBodies[0].angularVelocity = angularVelocity
if hasattr(config, 'rigidBodies'):
    schemeConfig.rigidBodies = config.rigidBodies


In [ ]:


schemeConfig.fluid.fixedSoundSpeed, config.dt = setupWeaklyCompressibleTimestep(config, schemeConfig, compressibleSystem, targetDt, verbose = True)
runningState = compressibleSystem.initializeNewState()

t_limit = 10.0
nSteps = int(t_limit / config.dt)


In [ ]:

markerSize = 8
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.velocities,
        "B":runningState.state.UIDs,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "velocities",
            boundaryVisualization= VisualizeOptions.Visualize,
            fluidVisualization= VisualizeOptions.Hide
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
            # vMin=0,
            # vMax=1.0
        ),
        "B": PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # colorMap = UniformColorMap.viridis,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "UIDs",
            boundaryVisualization= VisualizeOptions.Visualize,
            fluidVisualization= VisualizeOptions.Hide
            # vMin = 0.99,
            # vMax = 1.01
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'AB',
    figsize= (10,5),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

# imagePath = f'{exportPath}/images'
# os.makedirs(imagePath, exist_ok = True)
# plotter.export(f'{imagePath}/frame_00000.png', dpi = 300)



In [ ]:
t_limit = 10.0
nSteps = int(t_limit / config.dt)

runningState = compressibleSystem.initializeNewState()

kes = []
priorStep = None
for i in (tq := tqdm(range(nSteps), leave = False)):

    for rigidBody in schemeConfig.rigidBodies:
        rigidBody = integrateRigidBody(rigidBody, 0, 0, config.dt)
        runningState.state = updateBodyParticlesWCSPH(runningState.state, rigidBody)
    runningState.t += config.dt

    t = runningState.t
    if not isinstance(t, torch.Tensor):
        t = torch.tensor(t, device = device, dtype = dtype)
    if obstacleMotionType == 'fixed':
        linearVelocity = obstacleLinearVelocity
        angularVelocity = torch.tensor(obstacleAngularVelocity, device = device, dtype = dtype)
    elif obstacleMotionType == 'sinusoidal':
        linearVelocity =  obstacleLinearVelocity * torch.cos(t * np.pi * obstacleMotionFrequency)
        angularVelocity = obstacleAngularVelocity * torch.cos(t * np.pi * obstacleMotionFrequency) 

    if args.linearMotion == True:
        config.rigidBodies[0].linearVelocity = linearVelocity
        schemeConfig.rigidBodies[0].linearVelocity = linearVelocity
    if args.angularMotion == True:
        config.rigidBodies[0].angularVelocity = angularVelocity
        schemeConfig.rigidBodies[0].angularVelocity = angularVelocity

    # begin = torch.cuda.Event(enable_timing=True)
    # end = torch.cuda.Event(enable_timing=True)
    # begin.record()
    # result = integrator.function(
    #     state = runningState,
    #     f = fn,
    #     dt = config.dt,  
    #     config = config,
    #     schemeConfig = schemeConfig,
    #     verbose = False,
    #     # priorStep = priorStep
    # )
    # kes.append(torch.sum(0.5 * result.state.state.masses * torch.sum(result.state.state.velocities**2, dim=1)))
    # # print('max_vel:', torch.linalg.norm(result.state.state.velocities, dim = -1).max())
    # end.record()
    # torch.cuda.synchronize()
    # priorStep = result.stages[-1]
    # timing = begin.elapsed_time(end)

    # runningState = result.state

    # currentState = runningState.state
    # # print(f'-' * 80)
    # # print(f'Fluid density stats: min={currentState.densities[currentState.kinds == 0].min().item()}, max={currentState.densities[currentState.kinds == 0].max().item()}, mean={currentState.densities[currentState.kinds == 0].mean().item()}')
    # # print(f'Boundary density stats: min={currentState.densities[currentState.kinds == 1].min().item()}, max={currentState.densities[currentState.kinds == 1].max().item()}, mean={currentState.densities[currentState.kinds == 1].mean().item()}')
    if i % 100 == 0 :
        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                "B": runningState.state.UIDs,
            },
            newParticleState = runningState.state,
        )
        # plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
    # break
        
    # maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    # tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    # # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # # break
    # if torch.any(torch.isnan(runningState.state.velocities)):
    #     print("NaN detected in velocities, stopping simulation.")
    #     break



In [ ]:


schemeConfig.fluid.fixedSoundSpeed, config.dt = setupWeaklyCompressibleTimestep(config, schemeConfig, compressibleSystem, targetDt, verbose = True)

In [ ]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
dtype = get_torch_precision()


dx = L / (nx)
band = 0

domain = buildDomainDescription(L + dx * (band) * 2, dim, True, device, dtype)
interiorDomain = buildDomainDescription(L, dim, False, device, dtype)

config, integrator = buildConfig(
    domain = domain,
    dim = dim,
    kernel = KernelFunctions.Wendland4,
    targetNeighbors = n_h_to_nH(4, dim),
    supportMode = SupportScheme.KernelMeanSymmetric,
    gradientMode = GradientScheme.Difference,
    laplacianMode = LaplacianScheme.Brookshaw,
    integrationScheme = IntegrationSchemeType.rungeKutta2,
    samplingScheme = SamplingScheme.regular,
    device = device,
    dtype = dtype,
    dt = None,
    adaptiveDt = True,
    cflFactor=0.3,
)
config.dx = dx
config.nx = nx

config.minDt = 1e-8
# config.dx = L / (nx * 2)

scheme = WeaklyCompressibleSPHScheme.deltaSPH
bundle = buildScheme(scheme)
SimulationSystem, SimulationState = bundle.SimulationSystem, bundle.SimulationState
SimulationUpdate = bundle.SimulationUpdate
SimulationConfig = bundle.SimulationConfig
fn, export_fn, import_fn = bundle.stepFunction, bundle.exportFunction, bundle.importFunction


schemeConfig = SimulationConfig()
schemeConfig.surfaceDetectionConfig.active = freeSurface


In [ ]:
fluid_sdf = lambda x: sampleDomainSDF(x, domain, invert = True)
domain_sdf = lambda x: sampleDomainSDF(x, interiorDomain, invert = False)
obstacle_sdf = lambda points: sampleSDF(points, lambda x: getSDF('hexagon')['function'](x, torch.tensor(1/4).to(points.device)), invert = False)

regions = []

regions.append(buildRegion(config, schemeConfig, fluid_sdf, RegionType.Fluid, initialConditions = {}))
# regions.append(buildRegion(config, schemeConfig, domain_sdf, RegionType.Boundary, initialConditions = {}, kind = BCType.noSlip))
# if obstacle:
regions.append(buildRegion(config, schemeConfig, obstacle_sdf, RegionType.Boundary, initialConditions = {}, kind = BCType.constant))


for region in regions:
    region = filterRegion(region, regions)
config.regions = schemeConfig.regions = regions

In [ ]:
fig, axis = plt.subplots(1, 1, figsize=(5, 5), squeeze=False)
plotRegions(regions, axis[0,0], plotFluid = True, plotParticles = True)

rectangle = plt.Rectangle((domain.min[0].cpu().numpy(), domain.min[1].cpu().numpy()), domain.max[0].cpu().numpy() - domain.min[0].cpu().numpy(), domain.max[1].cpu().numpy() - domain.min[1].cpu().numpy(), fill=False, color='black', lw=2)
axis[0,0].add_patch(rectangle)

rectangleInterior = plt.Rectangle((interiorDomain.min[0].cpu().numpy(), interiorDomain.min[1].cpu().numpy()), interiorDomain.max[0].cpu().numpy() - interiorDomain.min[0].cpu().numpy(), interiorDomain.max[1].cpu().numpy() - interiorDomain.min[1].cpu().numpy(), fill=False, color='red', lw=2)
axis[0,0].add_patch(rectangleInterior)

import matplotlib.patches as patches

for ax in axis.flatten():
    ax.set_aspect('equal')
    ax.set_xlim(domain.min[0].item(), domain.max[0].item())
    ax.set_ylim(domain.min[1].item(), domain.max[1].item())

fig.tight_layout()

In [ ]:
compressibleSystem = initializeWeaklyCompressibleSimulation(regions, config, schemeConfig, SimulationSystem, SimulationState, verbose = True)

# compressibleSystem.state.positions = shuffleParticles(compressibleSystem.state, config, schemeConfig, 128, jitterAmount = 1.0)


schemeConfig.fluid.fixedSoundSpeed, config.dt = setupWeaklyCompressibleTimestep(config, schemeConfig, compressibleSystem, targetDt, verbose = True)
print(f"Computed timestep: {config.dt:.6g}, target timestep: {targetDt:.6g}, diff: {abs(config.dt - targetDt):.6g}")

In [ ]:
config.rigidBodies[0].angularVelocity = obstacle_omega
schemeConfig.rigidBodies = config.rigidBodies

In [ ]:
def periodicMeanFlowForcing(state, cfg, schemeCfg, positions, d, n, t, dt):
    force = torch.zeros_like(state.positions)
    fluid_mask = state.kinds == 0
    if torch.count_nonzero(fluid_mask) == 0:
        return force

    # Force only the domain-mean velocity so turbulent/wake fluctuations are not directly damped.
    u_mean = state.velocities[fluid_mask].mean(dim=0)
    ax = (U_target - u_mean[0]) / forcing_tau
    ay = (-u_mean[1]) / forcing_tau

    force[fluid_mask, 0] = state.masses[fluid_mask] * ax
    force[fluid_mask, 1] = state.masses[fluid_mask] * ay
    return force

periodicForcingBC = BoundaryCondition(
    type=BoundaryConditionType.dynamic,
    sdf=fluid_sdf,
    forcingFunctions=[periodicMeanFlowForcing],
)

schemeConfig.boundaryConditions = [periodicForcingBC]

In [ ]:
runningState = compressibleSystem.initializeNewState()

caseName = 'movingObstacle'
exportPath = prepExport(f'{caseName}', config, schemeConfig, scheme, export_fn)
exportSimulationSystem(exportPath, 'initialState', scheme, compressibleSystem, exportAdjacency = False, stages = None, exportStagesAdjacency = False, extraData = dict({
    'frame_num': 0,
}, **extraData))

In [ ]:

markerSize = 8
plotter = visualize(
    particleState = runningState.state,
    domain = config.domain,
    quantities = {
        "A": runningState.state.velocities,
        "B":runningState.state.UIDs,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = UniformColorMap.viridis,
            markerSize = markerSize,
            midPoint = 0.0,
            quantityScaling = PlotScaling.Linear,
            mapping = Mapping.L2Norm,
            plotTitle = "velocities",
            # boundaryVisualization= VisualizeOptions.Visualize,
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
            # vMin=0,
            # vMax=1.0
        ),
        "B": PlottingOptions(
            colorMap = CyclicColorMap.twilight,
            # colorMap = UniformColorMap.viridis,
            # flipColorMap=True,
            markerSize = markerSize,
            # midPoint = 1.0,
            quantityScaling = PlotScaling.Linear,
            plotTitle = "UIDs",
            # boundaryVisualization= VisualizeOptions.Visualize,
            fluidVisualization= VisualizeOptions.Visualize
            # vMin = 0.99,
            # vMax = 1.01
            # gridVisualization = GridVisualization(
            #     resolution = 512,
            # ),
        ),
    },
    figTitle = "Wave Equation Example",
    mosaic = 'AB',
    figsize= (10,5),
    backend='vispy',
    # backend='pyVista',
    # backendOptions = {
    #     # In notebooks, use trame for reliable live updates.
    #     'jupyter_backend': 'trame',
    # }
)

imagePath = f'{exportPath}/images'
os.makedirs(imagePath, exist_ok = True)
plotter.export(f'{imagePath}/frame_00000.png', dpi = 300)



In [ ]:
# Use explicit viscosity for better control of effective Reynolds number in periodic wake tests.
schemeConfig.diffusionParams.inviscid = False

# Hexagon scale in obstacle_sdf is 1/4, so a representative obstacle diameter is about 0.5.
D_obstacle = 0.5
Re_target = 250.0
schemeConfig.diffusionParams.viscidNu = U_target * D_obstacle / Re_target

# Slightly lower density diffusion can help preserve coherent vortices in long runs.
schemeConfig.diffusionParams.densityDelta = 0.05

nu = schemeConfig.diffusionParams.viscidNu if schemeConfig.diffusionParams.inviscid == False else alphaToNu(schemeConfig.diffusionParams.inviscidAlpha, schemeConfig.fluid.fixedSoundSpeed, compressibleSystem.state.supports.mean().cpu().item(), config.dim)
alpha = nuToAlpha(schemeConfig.diffusionParams.viscidNu, schemeConfig.fluid.fixedSoundSpeed, compressibleSystem.state.supports.mean().cpu().item(), config.dim) if schemeConfig.diffusionParams.inviscid == False else schemeConfig.diffusionParams.inviscidAlpha

print(f'Using inviscid: {schemeConfig.diffusionParams.inviscid}, nu: {nu:.6g}, alpha: {alpha:.6g}')

Re_D = U_target * D_obstacle / nu
print(f"Target U: {U_target:.6g}, obstacle D: {D_obstacle:.6g}, Reynolds number Re_D: {Re_D:.6g}")
if alpha < 0.01:
    print('Running with alpha < 0.01 may result in unstable simulations in inviscid mode.')
nu_limit = alphaToNu(0.01, schemeConfig.fluid.fixedSoundSpeed, compressibleSystem.state.supports.mean().cpu().item(), config.dim)
Re_limit = U_target * D_obstacle / nu_limit
print(f'Reynolds limit based on alpha = 0.01, nu = {nu_limit:.6g}, Re_D = {Re_limit:.6g}')

In [ ]:
t_limit = 10.0
nSteps = int(t_limit / config.dt)

runningState = compressibleSystem.initializeNewState()

kes = []
priorStep = None
for i in (tq := tqdm(range(nSteps), leave = False)):
    begin = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    begin.record()
    result = integrator.function(
        state = runningState,
        f = fn,
        dt = config.dt,  
        config = config,
        schemeConfig = schemeConfig,
        verbose = False,
        # priorStep = priorStep
    )
    kes.append(torch.sum(0.5 * result.state.state.masses * torch.sum(result.state.state.velocities**2, dim=1)))
    # print('max_vel:', torch.linalg.norm(result.state.state.velocities, dim = -1).max())
    end.record()
    torch.cuda.synchronize()
    priorStep = result.stages[-1]
    timing = begin.elapsed_time(end)

    runningState = result.state

    currentState = runningState.state
    # print(f'-' * 80)
    # print(f'Fluid density stats: min={currentState.densities[currentState.kinds == 0].min().item()}, max={currentState.densities[currentState.kinds == 0].max().item()}, mean={currentState.densities[currentState.kinds == 0].mean().item()}')
    # print(f'Boundary density stats: min={currentState.densities[currentState.kinds == 1].min().item()}, max={currentState.densities[currentState.kinds == 1].max().item()}, mean={currentState.densities[currentState.kinds == 1].mean().item()}')
    if i % 20 == 0 :
        plotter.updateQuantities(
            {
                "A": runningState.state.velocities,
                "B": runningState.state.UIDs,
            },
            newParticleState = runningState.state,
        )
        plotter.export(f'{imagePath}/frame_{i:05d}.png', dpi = 300)
    # break
        
    maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
    tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
    # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
    # break
    if torch.any(torch.isnan(runningState.state.velocities)):
        print("NaN detected in velocities, stopping simulation.")
        break



In [ ]:
# exportSimulationSystem(exportPath, f'finalState', scheme, runningState, exportAdjacency = False, stages = result.stages, exportStagesAdjacency = True, extraData = dict(**extraData, **{
#     'kineticEnergy': kineticEnergy,
#     'thermalEnergy': thermalEnergy,
#     'totalEnergy': totalEnergy,
#     'frame_num': i,
# }))

In [ ]:
ffmpeg_cmd = "ffmpeg -y -loglevel error -hide_banner -framerate 50 -f image2 -pattern_type glob -i 'frame_*.png' -c:v libx264 -pix_fmt yuv420p -b:v 10M output.mp4"
subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
# ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4  -vf "fps=50,scale=540:-1:flags=lanczos,palettegen" palette.png'
# subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)
# ffmpeg_cmd = 'ffmpeg -y -loglevel error -hide_banner -i output.mp4 -i palette.png -filter_complex "fps=25,scale=540:-1:flags=lanczos[x];[x][1:v]paletteuse" out.gif'
# subprocess.run(shlex.split(ffmpeg_cmd), check=True, cwd = imagePath)

# # now copy the output.mp4 and out.gif to the parent directory for easier access
shutil.copy(f'{imagePath}/output.mp4', f'{exportPath}/output.mp4')
# shutil.copy(f'{imagePath}/out.gif', f'{exportPath}/out.gif');

We can run a convergence test to see how the viscosity scales as well:

In [ ]:
nu_tests = np.logspace(-1, -5, base=10, num=10)
test_data = []

for nu in tqdm(nu_tests, desc="Testing viscosities", leave=False):
    t_limit = 2.0
    nSteps = int(t_limit / config.dt)

    runningState = compressibleSystem.initializeNewState()
    schemeConfig.diffusionParams.viscidNu = nu

    kes = []
    priorStep = None
    for i in (tq := tqdm(range(nSteps), leave = False)):
        begin = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        begin.record()
        result = integrator.function(
            state = runningState,
            f = fn,
            dt = config.dt,  
            config = config,
            schemeConfig = schemeConfig,
            verbose = False,
            priorStep = priorStep
        )
        kes.append(torch.sum(0.5 * result.state.state.masses * torch.sum(result.state.state.velocities**2, dim=1)))
        # print('max_vel:', torch.linalg.norm(result.state.state.velocities, dim = -1).max())
        end.record()
        torch.cuda.synchronize()
        priorStep = result.stages[-1]
        timing = begin.elapsed_time(end)

        runningState = result.state

        if i % 60 == 0 and i > 0:
            plotter.updateQuantities(
                {
                    "A": runningState.state.velocities,
                    "B": runningState.state.densities,
                },
                newParticleState = runningState.state,
            )
            
        maxVel = torch.linalg.norm(runningState.state.velocities, dim = -1).max()
        tq.set_description(f"Step {i+1}/{nSteps}, time: {(i+1)*config.dt:8.4g}/{t_limit:8.4g} | max vel: {maxVel:.3g} | iter time: {timing:.3f} ms")
        # t = {runningState.t:2f}, dt = {config.dt:.3g}, ptcls = {len(runningState.state.positions)}\nTotal Energy: {totalEnergy:.3g}, Kinetic Energy: {kineticEnergy:.3g}, Thermal Energy: {thermalEnergy:.3g}'
        # break
        if torch.any(torch.isnan(runningState.state.velocities)):
            print("NaN detected in velocities, stopping simulation.")
            breakts = np.arange(len(kes)) * config.dt.cpu().item()
    kineticEnergy = np.array([ke.cpu().item() for ke in kes])
    E_k0 = kineticEnergy[0]

    # Fit an effective viscosity from d/dt log(E_k) = -4 * (ktgv**2) * nu_eff
    mask = (ts > 0) & (kineticEnergy > 0)
    slope = np.polyfit(ts[mask], np.log(kineticEnergy[mask] / E_k0), 1)[0]
    nu_eff = -slope / (4 * ktgv**2)

    test_data.append({
        'nu': nu,
        'nu_eff': nu_eff,
        'nu_diff': abs(nu_eff - nu),
        'nu_rel_diff': abs(nu_eff - nu) / nu,
        'E_k0': E_k0,
        'kineticEnergy': kineticEnergy,
        'ts': ts,
    })


In [ ]:
fig, axis = plt.subplots(1, 2, figsize=(12, 5), squeeze=False)

nus = np.array([data['nu'] for data in test_data])
nu_effs = np.array([data['nu_eff'] for data in test_data])
axis[0,0].loglog(nus, nu_effs, marker='o', label='nu_eff')
axis[0,0].loglog(nus, nus, marker='o', label='nu')
axis[0,0].set_xlabel('Input Viscosity (nu)')
axis[0,0].set_ylabel('Effective Viscosity (nu_eff)')
axis[0,0].set_title('Effective Viscosity vs Input Viscosity')
axis[0,0].legend()

axis[0,1].loglog(nus, nu_effs / nus, marker='o')
axis[0,1].set_xlabel('Input Viscosity (nu)')
axis[0,1].set_ylabel('nu_eff / nu')
axis[0,1].set_title('Ratio of Effective to Input Viscosity')

fig.tight_layout()

fig.savefig('nu_eff_vs_nu.png', dpi=300)
